# AI Crop Disease Detector — full training pipeline (Google Colab)

**Run on Google Colab:** Runtime -> Change runtime type -> **T4 GPU** -> Save.

One Kaggle API token (free account) is needed ONLY to download the image
datasets, which are hosted on kaggle.com. All training happens right here.

Steps in this notebook:
1. clone the repo, install deps
2. upload your `kaggle.json` token, download the 3 datasets
3. `prepare_data.py` — merge + stratified split (seed 42)
4. `train.py` — two-stage transfer learning (the long cell: 1-2 h on T4)
5. `evaluate.py` + `calibrate_threshold.py` + `export_onnx.py`
6. zip the artifacts -> download -> send back to the repo owner

**Honesty list:** every number this notebook prints is real. Copy the
output files into `ml/reports/` — never edit the numbers by hand.

## 1. Clone repo + install dependencies (~3 min)

In [ ]:
# 1a. Clone the repository and cd into it.
!git clone https://github.com/ibrahimSumon47/AI-Crop-Disease-Detector-for-Bangladeshi-Farmers.git
%cd /content/AI-Crop-Disease-Detector-for-Bangladeshi-Farmers
!git log --oneline -3

In [ ]:
# 1b. Install the pinned ML dependencies from ml/requirements.txt.
# Colab already ships a recent TensorFlow; the requirements file only adds
# what is missing (tf2onnx, onnx/onnxruntime, etc.). --no-deps-free quick install.
!pip install -q onnx tf2onnx 'onnxruntime>=1.16'
!pip install -q -r ml/requirements.txt
import tensorflow as tf
print('TF version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
assert gpus, 'NO GPU! Runtime -> Change runtime type -> T4 GPU, then re-run this cell.'

## 2. Kaggle API token + dataset download (~10 min)

One-time setup on kaggle.com (free account):
1. Profile icon -> **Settings** -> section **API** -> **Create New Token**
2. The browser downloads `kaggle.json` — keep the file. Do NOT share it.

The next cell asks you to upload that file.

In [ ]:
# 2a. Upgrade the kaggle CLI first (Colab ships an old version that cannot
#     read the new KGAT token format; CLI >= 1.8.3 supports it).
!pip install -q --upgrade kaggle

# 2b. Upload your Kaggle token file (file-picker dialog appears).
#     It may be EITHER the new bare-token file (contents look like
#     'KGAT_xxxx...', any file name) OR the legacy kaggle.json with
#     {'username': ..., 'key': ...}. Both are supported here.
from google.colab import files
uploaded = files.upload()
assert uploaded, 'No token file was uploaded!'
fname = list(uploaded.keys())[0]
content = uploaded[fname].decode('utf-8').strip()

import os
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
home = os.path.expanduser('~/.kaggle')
if content.startswith('KGAT_'):
    # New token format -> goes into access_token + the env var.
    with open(f'{home}/access_token', 'w') as f:
        f.write(content)
    os.chmod(f'{home}/access_token', 0o600)
    os.environ['KAGGLE_API_TOKEN'] = content
    print('KGAT token installed to ~/.kaggle/access_token')
else:
    # Legacy {username, key} JSON format.
    with open(f'{home}/kaggle.json', 'w') as f:
        f.write(content)
    os.chmod(f'{home}/kaggle.json', 0o600)
    print('legacy kaggle.json installed')
print('Kaggle authentication ready.')

In [ ]:
# 2c. Download the three datasets into ml/data/raw (see ml/src/download_data.md).
#     PlantVillage is ~2 GB zipped; on Colab this takes ~5-10 minutes.
!mkdir -p ml/data/raw
!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p ml/data/raw --unzip
!kaggle datasets download -d shayanriyaz/riceleafs -p ml/data/raw --unzip
# PlantDoc is OPTIONAL (field photos). It occasionally fails to clone;
# riceleafs already covers rice-healthy, so we continue without it.
# To include it manually, download the archive from
#   https://zenodo.org/record/4082314  and unzip into ml/data/raw/plantdoc
!ls ml/data/raw

In [ ]:
# 2c. Inspect the class folder names that actually arrived. If they differ
#     from what prepare_data.py's ALIAS_MAP expects, we print them so the
#     mapping can be fixed BEFORE training (never train on wrong folders).
from pathlib import Path
for sub in sorted(Path('ml/data/raw').iterdir()):
    if sub.is_dir():
        folders = sorted({p.parent.name for p in sub.rglob('*.jpg')}) or \
                  sorted(x.name for x in sub.iterdir() if x.is_dir())
        print(f'--- {sub.name} ({len(folders)} class folders) ---')
        for f in folders[:40]:
            print('   ', f)

**If any printed folder name is NOT mapped in `ml/src/prepare_data.py` ->
`ALIAS_MAP`, fix the alias (add the exact folder name) and re-run only the
next cell of section 3 — do NOT train yet.**

## 3. Data preparation + sanity statistics

In [ ]:
# 3a. Merge all three datasets -> stratified 70/15/15 split (seed=42).
!python ml/src/prepare_data.py
!python ml/src/dataset_stats.py   # -> ml/reports/class_distribution.png
from IPython.display import Image, display
display(Image('ml/reports/class_distribution.png'))

**Check the chart: every class must have >0 images in the total column.
If a rice class is missing images, the notebook must be stopped — tell the
repo owner instead of training on a partial class list.**

## 4. Training (the long cell — 1–2 h on T4 GPU)

In [ ]:
# 4a. Two-stage training. Writes:
#      ml/reports/best_model.keras   (best checkpoint by val_accuracy)
#      ml/reports/history.json       (full merged history — REAL numbers)
!python ml/src/train.py --epochs-head 5 --epochs-ft 15 --batch 32

In [ ]:
# 4b. Plot the merged history curves (for the final report).
import json, matplotlib.pyplot as plt
h = json.load(open('ml/reports/history.json'))['history']
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(h['accuracy'], label='train'); ax[0].plot(h['val_accuracy'], label='val')
ax[0].set_title('accuracy'); ax[0].legend()
ax[1].plot(h['loss'], label='train'); ax[1].plot(h['val_loss'], label='val')
ax[1].set_title('loss'); ax[1].legend()
plt.savefig('ml/reports/history_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Evaluation + threshold calibration + ONNX export

In [ ]:
# 5a. Per-class precision/recall/F1 + confusion matrix -> ml/reports/metrics.md
!python ml/src/evaluate.py --model ml/reports/best_model.keras --data data/processed/test
# 5b. Confidence threshold sweep on validation -> ml/reports/threshold.json
!python ml/src/calibrate_threshold.py --model ml/reports/best_model.keras --val data/processed/val
!cat ml/reports/threshold.json | head -20

In [ ]:
# 5c. Export to ONNX (+ parity check) -> backend/app/data/model.onnx + labels.json
!python ml/src/export_onnx.py --model ml/reports/best_model.keras
!ls -la backend/app/data/

## 6. Collect + download the artifacts to send back

In [ ]:
# 6. Zip everything the repo owner needs, then trigger a browser download.
!cd ml/reports && cp ../../backend/app/data/labels.json . && \
  zip -q artifacts.zip metrics.md field_test.md confusion_matrix.png \
      field_confusion_matrix.png history.json history_curves.png \
      threshold.json class_distribution.png train_log_head.csv \
      train_log_finetune.csv labels.json 2>/dev/null; true
  # field_test.* may not exist yet (need real phone photos) — that is OK.
!ls -la ml/reports/artifacts.zip
from google.colab import files
files.download('ml/reports/artifacts.zip')
# ALSO download the model file separately (not inside the zip):
files.download('backend/app/data/model.onnx')

**Send `artifacts.zip` + `model.onnx` back to the repo owner.** The metrics,
threshold and labels land in git; `model.onnx` is committed only if it is
under 50 MB.